<a href="https://colab.research.google.com/github/Preetitamrakar-phd/GenAI_Hands-on/blob/main/RAG_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Two-layer evaluation approach for RAG systems
2. RETRIEVAL LAYER: Precision, Recall, F1 Score
3. GENERATION LAYER: Groundedness (Faithfulness), Response Completeness
4. Using LLM-as-judge for generation metrics
5. Creating comprehensive evaluation reports

In [2]:
import json
import os
import numpy as np
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv

/tmp/ipykernel_11290/723800189.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [3]:
# Load environment variables
load_dotenv()

True

In [4]:
print("\nLayer 1: RETRIEVAL (Precision, Recall, F1)")
print("Layer 2: GENERATION (Groundedness, Completeness)")


Layer 1: RETRIEVAL (Precision, Recall, F1)
Layer 2: GENERATION (Groundedness, Completeness)


PART 1: Setup - Load Data and Build RAG System

In [5]:
# Load tickets
with open('synthetic_tickets.json', 'r', encoding='utf-8') as f:
    tickets = json.load(f)
print(f"✓ Loaded {len(tickets)} support tickets")

✓ Loaded 20 support tickets


In [6]:
# Load evaluation queries with ground truth
with open('evaluation_queries.json', 'r', encoding='utf-8') as f:
    eval_queries = json.load(f)
print(f"✓ Loaded {len(eval_queries)} evaluation queries with ground truth")

✓ Loaded 15 evaluation queries with ground truth


In [7]:
eval_queries[:2]

[{'query_id': 'Q1',
  'question': 'How do I fix authentication failures after password reset?',
  'relevant_ticket_ids': ['TICK-001', 'TICK-011', 'TICK-014'],
  'category': 'Authentication',
  'reference_answer': 'Clear all active sessions and force re-authentication after password change. Ensure SAML signature algorithms match between your app and IdP (SHA-256 recommended). Extend session timeout to 2 hours with activity-based renewal to prevent premature logouts. Implement automatic session cleanup on password change.'},
 {'query_id': 'Q2',
  'question': 'What causes database connection timeout errors?',
  'relevant_ticket_ids': ['TICK-002', 'TICK-010'],
  'category': 'Database',
  'reference_answer': 'Database connection pool was sized too small for peak load. Increase max_connections parameter, add connection pooling monitoring, and optimize long-running queries that hold connections.'}]

In [8]:
# Build documents
documents = []
for ticket in tickets:
    content = f"""Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}"""

    doc = Document(
        page_content=content,
        metadata={
            'ticket_id': ticket['ticket_id'],
            'title': ticket['title'],
            'category': ticket['category']
        }
    )
    documents.append(doc)

In [9]:
# Initialize embeddings and LLM with timeout
embeddings = OpenAIEmbeddings(
    model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small'),
    request_timeout=30
)


In [10]:
# Use direct OpenAI client instead of LangChain ChatOpenAI (fixes connection pooling issue)
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'), timeout=30.0, max_retries=2)
chat_model = os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')


In [11]:
print("✓ OpenAI models initialized (30s timeout)")

✓ OpenAI models initialized (30s timeout)


In [12]:
# Test API connection
print("\nTesting OpenAI API connection...")
try:
    test_response = openai_client.chat.completions.create(
        model=chat_model,
        messages=[{"role": "user", "content": "Say 'OK'"}],
        max_tokens=5
    )
    print("✓ API connection successful")
except Exception as e:
    print(f"✗ API connection failed: {str(e)[:100]}")
    exit(1)


Testing OpenAI API connection...
✓ API connection successful


In [13]:
# Build vector store
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="eval_supportdesk"
)
print("✓ Vector store built with Chroma")

✓ Vector store built with Chroma


In [15]:
# Create simple RAG function
def generate_answer(query, k=3):
    """
    Retrieve top-k supporting documents and generate a grounded answer.

    This helper centralizes the "retrieve + generate" path used by both
    retrieval and generation evaluations, ensuring consistent behavior.
    """
    # Retrieve documents
    docs = vector_store.similarity_search(query, k=k)

    # Build context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Create prompt
    prompt = f"""You are a technical support assistant. Answer the question using ONLY the provided context.

Context:
{context}

Question: {query}

Answer (cite ticket IDs):"""

    # Generate answer using direct OpenAI client
    response = openai_client.chat.completions.create(
        model=chat_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    answer = response.choices[0].message.content

    return {'answer': answer, 'source_documents': docs}

print("✓ RAG function ready")

✓ RAG function ready


PART 2: RETRIEVAL LAYER EVALUATION

In [16]:
print("\nMetrics: Precision, Recall, F1 Score")
print("Measures: Are we retrieving the right documents?")


Metrics: Precision, Recall, F1 Score
Measures: Are we retrieving the right documents?


In [17]:
def calculate_retrieval_metrics(retrieved_ids, relevant_ids, k=3):
    """
    Calculate Precision, Recall, and F1 at k

    Precision@k = (relevant docs in top-k) / k
    Recall@k = (relevant docs in top-k) / (total relevant docs)
    F1@k = harmonic mean of Precision and Recall
    """
    # Restrict to top-k because retrieval quality is rank-sensitive.
    retrieved_set = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)

    # True positives: relevant docs that were retrieved
    tp = len(retrieved_set & relevant_set)

    # Precision: what % of retrieved docs are relevant?
    precision = tp / k if k > 0 else 0

    # Recall: what % of relevant docs were retrieved?
    recall = tp / len(relevant_set) if len(relevant_set) > 0 else 0

    # F1: harmonic mean balancing precision and recall
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'true_positives': tp,
        'retrieved_count': k,
        'relevant_count': len(relevant_set)
    }

In [18]:
# Evaluate retrieval for all queries
print("\nEvaluating retrieval for all queries...")
retrieval_results = []


Evaluating retrieval for all queries...


In [19]:
for idx, eval_query in enumerate(eval_queries, 1):
    query = eval_query['question']
    relevant_ids = eval_query['relevant_ticket_ids']

    print(f"  [{idx}/{len(eval_queries)}] Processing query...", end='\r')

    try:
        # Retrieve documents
        results = vector_store.similarity_search(query, k=5)
        retrieved_ids = [doc.metadata['ticket_id'] for doc in results]

        # Calculate metrics at k=3
        metrics = calculate_retrieval_metrics(retrieved_ids, relevant_ids, k=3)
        metrics['query_id'] = eval_query['query_id']
        metrics['question'] = query
        metrics['retrieved'] = retrieved_ids[:3]
        metrics['relevant'] = relevant_ids

        retrieval_results.append(metrics)
    except Exception as e:
        print(f"\n  ⚠ Error on query {idx}: {str(e)[:50]}")
        continue

print(f"\n✓ Completed retrieval evaluation for {len(retrieval_results)} queries")


✓ Completed retrieval evaluation for 15 queries


In [20]:
# Aggregate results
print("RETRIEVAL METRICS (Averaged across all queries)")

avg_precision = np.mean([r['precision'] for r in retrieval_results])
avg_recall = np.mean([r['recall'] for r in retrieval_results])
avg_f1 = np.mean([r['f1'] for r in retrieval_results])

print(f"\nPrecision@3: {avg_precision:.4f}")
print(f"  → What % of retrieved documents are actually relevant?")
print(f"  → Target: > 0.80 for production")

print(f"\nRecall@3:    {avg_recall:.4f}")
print(f"  → What % of all relevant documents did we find?")
print(f"  → Target: > 0.70 for production")

print(f"\nF1 Score@3:  {avg_f1:.4f}")
print(f"  → Balanced measure of retrieval quality")
print(f"  → Target: > 0.75 for production")

RETRIEVAL METRICS (Averaged across all queries)

Precision@3: 0.5556
  → What % of retrieved documents are actually relevant?
  → Target: > 0.80 for production

Recall@3:    0.9000
  → What % of all relevant documents did we find?
  → Target: > 0.70 for production

F1 Score@3:  0.6667
  → Balanced measure of retrieval quality
  → Target: > 0.75 for production


In [21]:
# Show per-query breakdown
print("PER-QUERY RETRIEVAL RESULTS (Sample)")

for i in range(min(3, len(retrieval_results))):
    result = retrieval_results[i]
    print(f"\n{result['query_id']}: {result['question'][:60]}...")
    print(f"  Relevant:  {result['relevant']}")
    print(f"  Retrieved: {result['retrieved']}")
    print(f"  Precision: {result['precision']:.2f} | Recall: {result['recall']:.2f} | F1: {result['f1']:.2f}")


PER-QUERY RETRIEVAL RESULTS (Sample)

Q1: How do I fix authentication failures after password reset?...
  Relevant:  ['TICK-001', 'TICK-011', 'TICK-014']
  Retrieved: ['TICK-001', 'TICK-016', 'TICK-011']
  Precision: 0.67 | Recall: 0.67 | F1: 0.67

Q2: What causes database connection timeout errors?...
  Relevant:  ['TICK-002', 'TICK-010']
  Retrieved: ['TICK-002', 'TICK-014', 'TICK-010']
  Precision: 0.67 | Recall: 1.00 | F1: 0.80

Q3: Why do international credit card payments fail?...
  Relevant:  ['TICK-003']
  Retrieved: ['TICK-003', 'TICK-016', 'TICK-013']
  Precision: 0.33 | Recall: 1.00 | F1: 0.50


PART 3: GENERATION LAYER EVALUATION

In [22]:
print("\nMetrics: Groundedness, Response Completeness")
print("Measures: Is the generated answer faithful and complete?")


Metrics: Groundedness, Response Completeness
Measures: Is the generated answer faithful and complete?


In [23]:
# Initialize OpenAI client for LLM-as-judge (reuse the same client)
client = openai_client

In [24]:
def evaluate_groundedness(answer, context_docs):
    """
    Groundedness (Faithfulness): Is the answer supported by the retrieved context?
    Uses LLM-as-judge to check if answer contains hallucinations

    Returns: score 0.0-1.0 (higher = more grounded)
    """
    # Judge receives ONLY retrieved evidence to test faithfulness.
    context = "\n\n".join([doc.page_content for doc in context_docs])

    prompt = f"""Evaluate if the ANSWER is fully supported by the CONTEXT. Check for hallucinations or unsupported claims.

CONTEXT:
{context}

ANSWER:
{answer}

Rate the groundedness from 0 to 10:

Provide:
1. Score (0-10)
2. Reasoning (one sentence)

Format:
Score: X
Reasoning: <explanation>"""

    try:
        response = client.chat.completions.create(
            model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            timeout=30
        )

        output = response.choices[0].message.content

        # Parse score
        try:
            score_line = [line for line in output.split('\n') if line.startswith('Score:')][0]
            score = float(score_line.split(':')[1].strip()) / 10.0  # Normalize to 0-1
        except:
            score = 0.5  # Default if parsing fails

        return {
            'score': score,
            'verdict': 'GROUNDED' if score >= 0.7 else 'PARTIAL' if score >= 0.4 else 'HALLUCINATED',
            'explanation': output
        }
    except Exception as e:
        print(f"\n    ⚠ Error evaluating groundedness: {str(e)[:50]}")
        return {'score': 0.5, 'verdict': 'ERROR', 'explanation': str(e)}

In [25]:
def evaluate_completeness(question, answer, reference_answer=None):
    """
    Response Completeness: Does the answer fully address the question?
    Uses LLM-as-judge to check if all aspects of the question are answered

    Returns: score 0.0-1.0 (higher = more complete)
    """
    # If a reference answer exists, completeness becomes relative to ideal coverage.
    if reference_answer:
        prompt = f"""Evaluate if the ANSWER fully addresses the QUESTION compared to the REFERENCE ANSWER.

QUESTION:
{question}

REFERENCE ANSWER:
{reference_answer}

GENERATED ANSWER:
{answer}

Rate the completeness from 0 to 10:

Provide:
1. Score (0-10)
2. Reasoning (one sentence)

Format:
Score: X
Reasoning: <explanation>"""
    else:
        prompt = f"""Evaluate if the ANSWER fully addresses the QUESTION.

QUESTION:
{question}

ANSWER:
{answer}

Rate the completeness from 0 to 10:

Provide:
1. Score (0-10)
2. Reasoning (one sentence)

Format:
Score: X
Reasoning: <explanation>"""

    try:
        response = client.chat.completions.create(
            model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            timeout=30
        )

        output = response.choices[0].message.content

        # Parse score
        try:
            score_line = [line for line in output.split('\n') if line.startswith('Score:')][0]
            score = float(score_line.split(':')[1].strip()) / 10.0  # Normalize to 0-1
        except:
            score = 0.5  # Default if parsing fails

        return {
            'score': score,
            'verdict': 'COMPLETE' if score >= 0.7 else 'PARTIAL' if score >= 0.4 else 'INCOMPLETE',
            'explanation': output
        }
    except Exception as e:
        print(f"\n    ⚠ Error evaluating completeness: {str(e)[:50]}")
        return {'score': 0.5, 'verdict': 'ERROR', 'explanation': str(e)}


In [26]:
# Evaluate generation for sample queries
print("\nEvaluating generation quality (sample queries)...")
print("This may take 30-60 seconds per query...\n")


Evaluating generation quality (sample queries)...
This may take 30-60 seconds per query...



In [27]:
generation_results = []

for idx, eval_query in enumerate(eval_queries[:2], 1):  # Evaluate first 2 to save time
    query = eval_query['question']
    reference = eval_query.get('reference_answer', None)

    print("-"*80)
    print(f"\n[{idx}/2] Query: {query}")

    try:
        # Generate answer using RAG
        print("  → Generating answer...")
        result = generate_answer(query)
        answer = result['answer']
        source_docs = result['source_documents']

        print(f"\nGenerated Answer:\n{answer}")

        # Evaluate groundedness
        print("\n  → Evaluating Groundedness...")
        groundedness = evaluate_groundedness(answer, source_docs)
        print(f"    Score: {groundedness['score']:.2f} - {groundedness['verdict']}")

        # Evaluate completeness
        print("\n  → Evaluating Completeness...")
        completeness = evaluate_completeness(query, answer, reference)
        print(f"    Score: {completeness['score']:.2f} - {completeness['verdict']}")

        generation_results.append({
            'query_id': eval_query['query_id'],
            'question': query,
            'answer': answer,
            'groundedness_score': groundedness['score'],
            'completeness_score': completeness['score']
        })
    except Exception as e:
        print(f"\n  ✗ Error processing query: {str(e)[:80]}")
        continue

print(f"\n✓ Completed generation evaluation for {len(generation_results)} queries")

--------------------------------------------------------------------------------

[1/2] Query: How do I fix authentication failures after password reset?
  → Generating answer...

Generated Answer:
To fix authentication failures after a password reset, you should clear all active sessions and force re-authentication. This issue was identified in Ticket ID: TICK-001, where the password hash algorithm was updated but session tokens weren't invalidated. Additionally, implement automatic session cleanup on password change to prevent future occurrences.

  → Evaluating Groundedness...
    Score: 1.00 - GROUNDED

  → Evaluating Completeness...
    Score: 0.80 - COMPLETE
--------------------------------------------------------------------------------

[2/2] Query: What causes database connection timeout errors?
  → Generating answer...

Generated Answer:
Database connection timeout errors are caused by the connection pool being exhausted, as indicated in Ticket ID: TICK-002. The logs showed '

In [28]:
# Aggregate generation metrics
avg_groundedness = np.mean([r['groundedness_score'] for r in generation_results])
avg_completeness = np.mean([r['completeness_score'] for r in generation_results])

print(f"\nGroundedness:  {avg_groundedness:.4f}")
print(f"  → Are answers supported by retrieved context?")
print(f"  → Target: > 0.80 (minimize hallucinations)")

print(f"\nCompleteness:  {avg_completeness:.4f}")
print(f"  → Do answers fully address the questions?")
print(f"  → Target: > 0.75 (comprehensive responses)")


Groundedness:  1.0000
  → Are answers supported by retrieved context?
  → Target: > 0.80 (minimize hallucinations)

Completeness:  0.8000
  → Do answers fully address the questions?
  → Target: > 0.75 (comprehensive responses)


PART 4: Comprehensive Evaluation Report

In [29]:
print(f"""
┌─────────────────────────────────────────────────────────────┐
│                    RAG SYSTEM EVALUATION                    │
├─────────────────────────────────────────────────────────────┤
│ Dataset: {len(eval_queries)} evaluation queries                           │
├─────────────────────────────────────────────────────────────┤
│ RETRIEVAL LAYER                                             │
│   • Precision@3:  {avg_precision:.4f}  {'✓' if avg_precision >= 0.80 else '⚠' if avg_precision >= 0.70 else '✗'}                              │
│   • Recall@3:     {avg_recall:.4f}  {'✓' if avg_recall >= 0.70 else '⚠' if avg_recall >= 0.60 else '✗'}                              │
│   • F1 Score@3:   {avg_f1:.4f}  {'✓' if avg_f1 >= 0.75 else '⚠' if avg_f1 >= 0.65 else '✗'}                              │
├─────────────────────────────────────────────────────────────┤
│ GENERATION LAYER                                            │
│   • Groundedness: {avg_groundedness:.4f}  {'✓' if avg_groundedness >= 0.80 else '⚠' if avg_groundedness >= 0.70 else '✗'}                              │
│   • Completeness: {avg_completeness:.4f}  {'✓' if avg_completeness >= 0.75 else '⚠' if avg_completeness >= 0.65 else '✗'}                              │
└─────────────────────────────────────────────────────────────┘

INTERPRETATION:
""")


┌─────────────────────────────────────────────────────────────┐
│                    RAG SYSTEM EVALUATION                    │
├─────────────────────────────────────────────────────────────┤
│ Dataset: 15 evaluation queries                           │
├─────────────────────────────────────────────────────────────┤
│ RETRIEVAL LAYER                                             │
│   • Precision@3:  0.5556  ✗                              │
│   • Recall@3:     0.9000  ✓                              │
│   • F1 Score@3:   0.6667  ⚠                              │
├─────────────────────────────────────────────────────────────┤
│ GENERATION LAYER                                            │
│   • Groundedness: 1.0000  ✓                              │
│   • Completeness: 0.8000  ✓                              │
└─────────────────────────────────────────────────────────────┘

INTERPRETATION:



In [30]:
# ── Failure Pattern Playbook ─────────────────────────────────────────────────
# Match metrics against the four named diagnostic patterns.
if avg_precision >= 0.75 and avg_recall < 0.60:
    print("⚠ PATTERN A — High Precision, Low Recall")
    print("  Retrieval is too conservative: correct docs found but many missed.")
    print("  Fix: Increase k, use smaller chunks, expand query with synonyms.")

if avg_precision < 0.60 and avg_recall >= 0.75:
    print("⚠ PATTERN B — Low Precision, High Recall")
    print("  Too many irrelevant docs entering the result set.")
    print("  Fix: Use MMR, add metadata filters, improve embeddings.")

if avg_f1 >= 0.75 and (avg_groundedness < 0.70 or avg_completeness < 0.65):
    print("⚠ PATTERN C — Strong Retrieval, Weak Generation")
    print("  Right documents retrieved but LLM is not using them well.")
    print("  Fix: Strengthen prompt, add few-shot examples, use stronger LLM.")

if avg_groundedness < 0.70:
    print("⚠ PATTERN D — Low Groundedness (Hallucination Risk)")
    print("  LLM is generating claims not supported by retrieved context.")
    print("  Fix: Stricter grounding prompt, temperature=0, require ticket citations.")


⚠ PATTERN B — Low Precision, High Recall
  Too many irrelevant docs entering the result set.
  Fix: Use MMR, add metadata filters, improve embeddings.


In [31]:
# ── Release Gate ──────────────────────────────────────────────────────────────
# Determines whether this build should be deployed, reviewed, or blocked.
thresholds = {
    'precision':    {'pass': 0.80, 'review': 0.70},
    'recall':       {'pass': 0.70, 'review': 0.60},
    'f1':           {'pass': 0.75, 'review': 0.65},
    'groundedness': {'pass': 0.85, 'review': 0.75},
    'completeness': {'pass': 0.75, 'review': 0.65},
}
current_metrics = {
    'precision': avg_precision, 'recall': avg_recall, 'f1': avg_f1,
    'groundedness': avg_groundedness, 'completeness': avg_completeness,
}

In [32]:
red_flags, yellow_flags = [], []
for metric, value in current_metrics.items():
    t = thresholds[metric]
    if value < t['review']:
        red_flags.append(f"{metric} = {value:.2f} (min {t['review']})")
    elif value < t['pass']:
        yellow_flags.append(f"{metric} = {value:.2f} (target {t['pass']})")

In [33]:
if red_flags:
    decision = "BLOCK  — Do not deploy. Fix critical metrics first."
elif yellow_flags:
    decision = "REVIEW — Investigate before deploying."
else:
    decision = "PASS   — All metrics in target range. Ready to deploy."

In [34]:
print("RELEASE GATE DECISION")

print(f"  {decision}")
if red_flags:
    print("\n  Critical (RED):")
    for f in red_flags:
        print(f"    ✗ {f}")
if yellow_flags:
    print("\n  Warning (YELLOW):")
    for f in yellow_flags:
        print(f"    ⚠ {f}")
print("-"*60)


RELEASE GATE DECISION
  BLOCK  — Do not deploy. Fix critical metrics first.

  Critical (RED):
    ✗ precision = 0.56 (min 0.7)

  Warning (YELLOW):
    ⚠ f1 = 0.67 (target 0.75)
------------------------------------------------------------


PART 5: Comparing Different RAG Configurations

In [37]:
print("PART 5: A/B Testing Framework")
print("\nExample: Compare retrieval with k=3 vs k=5")

PART 5: A/B Testing Framework

Example: Compare retrieval with k=3 vs k=5


In [38]:
def compare_configurations(queries, k_values):
    """
    Compare retrieval settings (e.g., different k values) on the same query set.

    This is a lightweight A/B harness that helps choose configuration defaults
    based on measured precision/recall/F1 instead of intuition.
    """
    results = {}

    for k in k_values:
        print(f"  Testing k={k}...")
        retrievals = []
        for query in queries[:3]:  # Test subset
            try:
                docs = vector_store.similarity_search(query['question'], k=k)
                retrieved_ids = [doc.metadata['ticket_id'] for doc in docs]
                metrics = calculate_retrieval_metrics(retrieved_ids, query['relevant_ticket_ids'], k=k)
                retrievals.append(metrics)
            except Exception as e:
                print(f"\n    ⚠ Error: {str(e)[:50]}")
                continue

        if retrievals:
            results[f'k={k}'] = {
                'precision': np.mean([r['precision'] for r in retrievals]),
                'recall': np.mean([r['recall'] for r in retrievals]),
                'f1': np.mean([r['f1'] for r in retrievals])
            }

    return results

In [41]:
comparison = compare_configurations(eval_queries, [3, 5])
print("\nConfiguration Comparison:")
print(f"{'Config':<10} {'Precision':<12} {'Recall':<12} {'F1':<12}")
for config, metrics in comparison.items():
    print(f"{config:<10} {metrics['precision']:<12.4f} {metrics['recall']:<12.4f} {metrics['f1']:<12.4f}")

  Testing k=3...
  Testing k=5...

Configuration Comparison:
Config     Precision    Recall       F1          
k=3        0.5556       0.8889       0.6556      
k=5        0.4000       1.0000       0.5516      
